# Lab F1. Loss functions and the properties of a good forecast

A forecast is good or bad only under a loss function. This lab simulates an economy whose true model we know, so
every answer can be checked: what the best forecast is under squared and under asymmetric loss, and whether a
forecast is efficient (Mincer-Zarnowitz).

**Help:** for any pymetrix function, type its name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call. The last cell lists every pymetrix function, class and method this lab uses.

Topics: loss functions; the optimal forecast under squared, lin-lin and linex loss; forecast efficiency; the
pseudo out-of-sample backtest (Elliott and Timmermann, chapters 2, 3, 15 and 16).

The data: 240 quarters of an AR(1), $y_t = 1 + 0.6\,y_{t-1} + \varepsilon_t$ with $\varepsilon_t \sim N(0, 1)$.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from pymetrix import Backtest, Dataset, Equation, backtest
from pymetrix.stats import linex_loss, linlin_loss, mincer_zarnowitz

rng = np.random.default_rng(2016)
n, c, phi = 240, 1.0, 0.6
y = np.zeros(n)
y[0] = c / (1 - phi)
for t in range(1, n):
    y[t] = c + phi * y[t - 1] + rng.standard_normal()
data = Dataset.from_frame(pd.DataFrame({"y": y}, index=pd.period_range("1965Q1", periods=n, freq="Q")), source="simulated AR(1)")
data

## 1. The forecasts and their errors

A backtest re-estimates the model at each origin from 2000Q1 to 2023Q4, using only the data to that origin, and
forecasts one to four quarters ahead. With the true model, the $h$-step error has variance
$\sigma^2 (1 + \phi^2 + \dots + \phi^{2(h-1)})$: the RMSEs should be close to those numbers (the parameters are
estimated, so a little above).

In [ ]:
ar = Equation("y ~ y[-1]").fit(data)
bt = backtest(ar, data, ("2000Q1", "2023Q4"), horizon=4, label="AR(1)")
theory = pd.Series({h: np.sqrt(sum(phi ** (2 * k) for k in range(h))) for h in range(1, 5)}, name="true RMSE")
scores = bt.scores(["y"])
pd.DataFrame({"backtest RMSE": scores["RMSE"].to_numpy(), "true RMSE": theory.to_numpy()}, index=theory.index.rename("horizon"))

## 2. Squared loss and asymmetric loss

Under squared loss the best forecast is the conditional mean. Under lin-lin loss with $\tau = 0.8$, an
under-forecast costs four times as much as an over-forecast of the same size, and the best forecast is the
conditional 80% quantile: the mean plus $\Phi^{-1}(0.8)$ times the forecast's standard deviation.

Make that forecast from the backtest's own errors: shift each mean forecast up by $\Phi^{-1}(0.8)$ times its
horizon's RMSE from the past forecasts (here, for simplicity, the true RMSE).

In [ ]:
tau = 0.8
shifted = bt.errors.copy()
shifted["forecast"] += stats.norm.ppf(tau) * shifted["horizon"].map(theory)
shifted["error"] = shifted["actual"] - shifted["forecast"]
quantile = Backtest(shifted, label="80% quantile")

table = pd.DataFrame({
    "squared loss": [bt.losses("y", 1).mean(), quantile.losses("y", 1).mean()],
    "lin-lin loss (tau = 0.8)": [bt.losses("y", 1, "lin-lin", tau=tau).mean(), quantile.losses("y", 1, "lin-lin", tau=tau).mean()],
}, index=["mean forecast", "80% quantile forecast"])
table

Each forecast wins under its own loss. The Diebold-Mariano test asks whether the difference is more than chance
(a negative statistic favours the first forecast).

In [ ]:
under_squared = bt.compare(quantile, "y", 1)
under_linlin = quantile.compare(bt, "y", 1, loss="lin-lin", tau=tau)
{"squared: mean vs quantile": (round(float(under_squared.statistic), 2), round(float(under_squared.p_value), 4)),
 "lin-lin: quantile vs mean": (round(float(under_linlin.statistic), 2), round(float(under_linlin.p_value), 4))}

**Check it yourself.** The Diebold-Mariano statistic is the mean loss differential over its standard error,
here (one step ahead) with the Harvey-Leybourne-Newbold small-sample factor $\sqrt{(n - 1)/n}$ and a Student t
with $n - 1$ degrees of freedom.

In [ ]:
d = (bt.losses("y", 1) - quantile.losses("y", 1)).to_numpy()
m = len(d)
by_hand = d.mean() / np.sqrt(d.var() / m) * np.sqrt((m - 1) / m)
assert np.isclose(by_hand, under_squared.statistic)
assert np.isclose(2 * stats.t.sf(abs(by_hand), m - 1), under_squared.p_value)
print(f"DM by hand: {by_hand:.4f}; pymetrix: {under_squared.statistic:.4f}")

Under linex loss with $a > 0$, under-forecasts cost exponentially. For a normal forecast the best forecast is the
mean plus $a\sigma^2/2$. Search for the best constant shift of the one-step forecasts under each loss: the
lin-lin shift should be near $\Phi^{-1}(0.8) = 0.84$ and the linex shift (with $a = 1$) near $1/2$.

In [ ]:
e1 = bt.errors.query("horizon == 1")["error"].to_numpy()
shifts = np.linspace(-1, 2, 301)
best_linlin = shifts[np.argmin([linlin_loss(e1 - s, tau).mean() for s in shifts])]
best_linex = shifts[np.argmin([linex_loss(e1 - s, 1.0).mean() for s in shifts])]
{"best lin-lin shift": round(float(best_linlin), 2), "theory": round(float(stats.norm.ppf(tau)), 2),
 "best linex shift": round(float(best_linex), 2), "theory (a = 1, sigma = 1)": 0.5}

**Question.** The best shifts found from the backtest are not exactly the theoretical ones. Why not? Would they
get closer with more forecasts?

## 3. Is the forecast efficient?

Under squared loss an efficient forecast leaves nothing predictable in its errors. Mincer and Zarnowitz regress
the outcome on the forecast, $y_{t+h} = a + b\,\hat y_{t+h|t} + u$, and test $a = 0$, $b = 1$.

Compare the AR(1) forecasts with a naive forecast, the last observation ("no change"). The true process reverts
to its mean, so the naive forecast moves too much: its slope is below one.

In [ ]:
naive = bt.errors.copy()
last = data["y"]
naive["forecast"] = [float(last[pd.Period(o, freq="Q")]) for o in naive["origin"]]
naive["error"] = naive["actual"] - naive["forecast"]
naive = Backtest(naive, label="no change")
mz_ar, mz_naive = bt.efficiency("y", 1), naive.efficiency("y", 1)
pd.DataFrame({"intercept": [mz_ar.details["intercept"], mz_naive.details["intercept"]],
              "slope": [mz_ar.details["slope"], mz_naive.details["slope"]],
              "p-value (a = 0, b = 1)": [mz_ar.p_value, mz_naive.p_value]}, index=["AR(1)", "no change"])

**Check it yourself.** One step ahead the test is a Wald test of $(a, b) = (0, 1)$ with White standard errors:
statsmodels gives the same statistic.

In [ ]:
import statsmodels.api as sm

one = naive.errors.query("horizon == 1")
ols = sm.OLS(one["actual"].to_numpy(), sm.add_constant(one["forecast"].to_numpy())).fit(cov_type="HC0")
wald = ols.wald_test((np.eye(2), np.array([0.0, 1.0])), use_f=False, scalar=True)
assert np.isclose(float(wald.statistic), mz_naive.statistic)
print(f"Wald by statsmodels: {float(wald.statistic):.4f}; pymetrix: {mz_naive.statistic:.4f}")

**Questions.**

1. The efficiency test at four quarters uses Newey-West standard errors with three lags. Why do $h$-step errors
   overlap, and what would go wrong with White standard errors? (Try `bt.efficiency("y", 4)` and
   `mincer_zarnowitz(actual, forecast, horizon=1)` on the same four-step forecasts.)
2. Is the 80% quantile forecast efficient by the Mincer-Zarnowitz test? Should it be?

In the app: **Forecast → Backtest** makes `bt`; **Forecast → Compare backtests** runs the Diebold-Mariano test with
any of the losses (lin-lin's $\tau$ and linex's $a$ are on its Advanced tab); a backtest's window has an
Evaluation tab with the Mincer-Zarnowitz tests.

In [ ]:
four = bt.errors.query("horizon == 4")
nw, white = bt.efficiency("y", 4), mincer_zarnowitz(four["actual"], four["forecast"], horizon=1)
{"slope se, Newey-West (3 lags)": round(nw.details["slope se"], 3), "slope se, White": round(white.details["slope se"], 3)}

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `Backtest(errors, label, provenance, weights)` | Forecasts from each origin, with what happened: one row per origin, horizon and variable. |
| `backtest(fitted, data, origins, horizon, vintages, variable, label, window)` | Re-estimate at each origin from `origins[0]` to `origins[1]` and forecast `horizon` periods ahead. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |

### From `pymetrix.stats`

| | |
|---|---|
| `linex_loss(errors, a)` | Linex loss (Varian 1975) of errors e = outcome - forecast: exp(a e) - a e - 1. |
| `linlin_loss(errors, tau)` | Lin-lin (check, or pinball) loss of errors e = outcome - forecast: tau e when e >= 0, (tau - 1) e otherwise. |
| `mincer_zarnowitz(actual, forecast, horizon)` | Mincer-Zarnowitz efficiency test: regress the outcome on a constant and the forecast, and test a = 0, b = 1 (Wald, chi-squared with 2 degrees of freedom; Newey-West standard errors with h - 1 lags). |

### Methods and properties of the results

| | |
|---|---|
| `Backtest.compare(other, variable, horizon, loss, test, tau, a)` | Test whether these forecasts and `other`'s are equally accurate at one horizon, on the targets both forecast. |
| `Backtest.efficiency(variable, horizon)` | Mincer-Zarnowitz test that the forecasts are efficient: outcome = a + b forecast, with a = 0 and b = 1. |
| `Backtest.losses(variable, horizon, loss, tau, a)` | The loss of each forecast of `variable` at `horizon`, by target period: the `"squared"` or `"absolute"` error, the asymmetric `"lin-lin"` (with `tau`) or `"linex"` (with `a`) loss of the error, the `"crps"`, or the negative `"log score"` (so that lower is better for all of them). |
| `Backtest.scores(variables)` | n, RMSE, MAE and mean error by variable and horizon (a two-level index). |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source, start)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |